In [ ]:
import json
import csv
import io
from datetime import datetime
from typing import List, Union, Optional, Dict
from dataclasses import dataclass

@dataclass(frozen=True)
class IngestedTelemetryRecord:
    """Standardized telemetry record."""
    timestamp: datetime        # ISO 8601
    rack_temps: List[float]    # °C
    airflow_proxy: float       # % or RPM
    chiller_power: float       # kW
    facility_power: float      # kW
    ups_load: float            # kW or %
    tariff: float              # $/kWh
    raw_format: str            # 'json' or 'csv'

class EnerviaIngestion:
    """Data Ingestion Service."""

    def __init__(self):
        # Constants
        pass

    def _parse_timestamp(self, ts_raw: Union[str, int, float]) -> datetime:
        """Standardize timestamp."""
        if isinstance(ts_raw, (int, float)):
            return datetime.fromtimestamp(ts_raw)
        try:
            return datetime.fromisoformat(ts_raw)
        except ValueError:
            # Fallback format
            return datetime.strptime(ts_raw, "%Y-%m-%d %H:%M:%S")

    def _normalize_rack_temps(self, temps: Union[float, List[float]]) -> List[float]:
        """Normalize temps to list."""
        if isinstance(temps, (int, float)):
            return [float(temps)]
        return [float(t) for t in temps]

    def process_incoming_payload(self, payload: str) -> IngestedTelemetryRecord:
        """Process payload."""
        payload = payload.strip()
        
        if payload.startswith('{'):
            return self._handle_json(payload)
        else:
            return self._handle_csv(payload)
        

    def _handle_json(self, payload: str) -> IngestedTelemetryRecord:
        """Parse JSON."""
        data = json.loads(payload)
        
        return IngestedTelemetryRecord(
            timestamp=self._parse_timestamp(data['timestamp']),
            rack_temps=self._normalize_rack_temps(data['rack_temps']),
            airflow_proxy=float(data['airflow_proxy']),
            chiller_power=float(data['chiller_power']),
            facility_power=float(data['facility_power']),
            ups_load=float(data['ups_load']),
            tariff=float(data['tariff']),
            raw_format='json'
        )

    def _handle_csv(self, payload: str) -> IngestedTelemetryRecord:
        """Parse CSV."""
        f = io.StringIO(payload)
        reader = csv.DictReader(f)
        row = next(reader)
        
        # Parse temps
        raw_temps = row['rack_temps']
        if ';' in raw_temps:
            processed_temps = [float(t) for t in raw_temps.split(';')]
        else:
            processed_temps = [float(raw_temps)]

        return IngestedTelemetryRecord(
            timestamp=self._parse_timestamp(row['timestamp']),
            rack_temps=processed_temps,
            airflow_proxy=float(row['airflow_proxy']),
            chiller_power=float(row['chiller_power']),
            facility_power=float(row['facility_power']),
            ups_load=float(row['ups_load']),
            tariff=float(row['tariff']),
            raw_format='csv'
        )


# ingestion_service = EnerviaIngestion()
# record = ingestion_service.process_incoming_payload(raw_data_string)